# Probabilistic Logic Demo: Pulling Predicates at Scale

This notebook demonstrates the power of extracting predicates at scale and applying probabilistic logic to them. We utilize **Distributional Semantics** to derive truth confidence from the frequency of assertions in a text corpus.

We will:
1.  Load ~20k filtered predicates.
2.  Merge specific entities (e.g., 'War' and 'war') to consolidate knowledge.
3.  Build a **Sparse LogicModel** capable of handling thousands of elements.
4.  Derive probabilities: $P(predicate) = \frac{count(predicate)}{max\_count(subject)}$.
5.  Explore **6 Case Studies** answering complex questions using logical conjunctions (AND) and disjunctions (OR).

In [1]:
import pickle
import LogicModel as m
import numpy as np
import collections
import scipy.sparse as sp

# 1. Load Data
try:
    with open('Predicates/FILTERED-predicates.pickle', 'rb') as f:
        raw_pos = pickle.load(f)
    print(f"Loaded {len(raw_pos)} filtered positive predicates.")
    
    with open('Predicates/FILTERED-negative_predicates.pickle', 'rb') as f:
        raw_neg = pickle.load(f)
    print(f"Loaded {len(raw_neg)} filtered negative predicates.")
except FileNotFoundError:
    print("Pickle files not found.")
    raw_pos = []
    raw_neg = []

Loaded 30248 filtered positive predicates.
Loaded 30145 filtered negative predicates.


## 2. Preprocessing & Merging

We filter out common pronouns to focus on substantive entities. We also explicitly merge **'War'** into **'war'** to combine the knowledge base for this concept.

In [2]:
pronouns = {'he', 'she', 'it', 'him', 'her', 'they', 'them', 'we', 'us', 'you',
            'He', 'She', 'It', 'Him', 'Her', 'They', 'Them', 'We', 'Us', 'You',
            'this', 'This', 'that', 'That', 'these', 'These', 'those', 'Those', 
            'I', 'me', 'Me', 'my', 'My', 'myself', 'Myself',
            'which', 'Which', 'one', 'One'}

def clean_and_merge(pred_list):
    cleaned = []
    for s, p, o in pred_list:
        # Filter Pronouns
        if s in pronouns: continue
        if o is not None and o in pronouns: continue
        
        # Merge 'War' -> 'war'
        if s == 'War': s = 'war'
        if o == 'War': o = 'war'
        
        cleaned.append((s, p, o))
    return cleaned

clean_pos = clean_and_merge(raw_pos)
clean_neg = clean_and_merge(raw_neg)

print(f"Total cleaned positive facts: {len(clean_pos)}")
print(f"Total cleaned negative facts: {len(clean_neg)}")

Total cleaned positive facts: 17076
Total cleaned negative facts: 18632


## 3. Build LogicModel (Sparse)

We construct the domain and initialize the logic model using sparse matrices for efficiency.

In [3]:
domain_set = set()
unary_preds_dict = {}
binary_preds_dict = {}

def add_to_domain(elem):
    if elem is not None:
        domain_set.add(elem)

# Process Positive (True facts)
for s, p, o in clean_pos:
    add_to_domain(s)
    add_to_domain(o)
    
    if o is None:
        if p not in unary_preds_dict: unary_preds_dict[p] = []
        unary_preds_dict[p].append(s)
    else:
        if p not in binary_preds_dict: binary_preds_dict[p] = []
        binary_preds_dict[p].append((s, o))

# Process Negative (False facts) - Initialize keys
for s, p, o in clean_neg:
    add_to_domain(s)
    add_to_domain(o)
    if o is None:
        if p not in unary_preds_dict: unary_preds_dict[p] = []
        unary_preds_dict[p].append((s, 0.0)) # Explicit False
    else:
        if p not in binary_preds_dict: binary_preds_dict[p] = []

domain_list = sorted(list(domain_set))

print("Building sparse model...")
model = m.LogicModel(
    listOfElements=domain_list,
    dictionaryOfUnaryPredicates=unary_preds_dict,
    dictionaryOfBinaryPredicates=binary_preds_dict,
    use_sparse=True
)
model.buildAll()
print("Model built!")

Building sparse model...


Model built!


## 4. Helper Function: Update Probabilities & Query

This function calculates probabilities for a subject based on frequency and executes a query.

In [4]:
def analyze_subject(subject, predicates_of_interest, query_logic_fn=None):
    """
    1. Counts predicate occurrences for the subject.
    2. Updates the model with derived probabilities.
    3. Runs the specific logic query provided.
    """
    print(f"\n--- Analysis: {subject} ---")
    
    # 1. Count Frequencies
    predicate_counts = collections.Counter()
    for s, p, o in clean_pos:
        if s == subject:
            predicate_counts[(p, o)] += 1
            
    if not predicate_counts:
        print(f"No facts found for {subject}")
        return

    max_count = predicate_counts.most_common(1)[0][1]
    print(f"Max frequency count: {max_count}")
    
    # 2. Update Model Probabilities
    for (pred, obj), count in predicate_counts.items():
        prob = count / max_count
        # Only print if it's one of our interest predicates to keep output clean
        if pred in predicates_of_interest:
            print(f"  Fact: {pred}({subject}, {obj}) | Count: {count} | Prob: {prob:.2f}")
        
        if obj is None:
            model.updateUnaryPredicate(subject, pred, prob)
        else:
            model.updateBinaryPredicate((subject, obj), pred, prob)
            
    # 3. Execute Query
    if query_logic_fn:
        query_logic_fn(subject)


## Case Study 1: Bernie Sanders (Political)
**Questions:**
*   Is Sanders a socialist?
*   Is Sanders a member?
*   **Logic**: Is Sanders a socialist AND a member?

In [5]:
def query_sanders(subj):
    p1 = "is_socialist"
    p2 = "is_member"
    
    val1 = model.unaryOp(p1, subj)
    val2 = model.unaryOp(p2, subj)
    
    print(f"\n1. {p1}({subj}): \n{val1.flatten()}")
    print(f"2. {p2}({subj}): \n{val2.flatten()}")
    
    res = model.andOp(val1, val2)
    print(f"3. {p1} AND {p2}: \n{res.flatten()}")

analyze_subject("Sanders", ["is_socialist", "is_member"], query_sanders)


--- Analysis: Sanders ---
Max frequency count: 2
  Fact: is_member(Sanders, None) | Count: 2 | Prob: 1.00


  Fact: is_socialist(Sanders, None) | Count: 1 | Prob: 0.50

1. is_socialist(Sanders): 
[0.5 0.5]
2. is_member(Sanders): 
[1. 0.]
3. is_socialist AND is_member: 
[0.5 0.5]


## Case Study 2: Napoleon (History)
**Questions:**
*   Did Napoleon return?
*   Did Napoleon escape?
*   **Logic**: Did Napoleon return AND escape?

In [6]:
def query_napoleon(subj):
    # Note: These are binary predicates in the data with 'None' or 'obj'? 
    # Checking data, they often appear as binaries with objects, but we'll check the unary projection if obj is None
    # Or simply finding ANY matching predicate.
    # For simplicity in this demo, we assume (Subject, Predicate, None) for Unary checks usually,
    # but let's check binaries if 'return' takes an object.
    
    # Since our analyze_subject updates whatever (pred, obj) pair exists,
    # we need to be careful. 'return' might be 'return(Napoleon, France)'.
    # For this demo, let's construct Unary vectors from the specific (Pred, Obj) pairs found.
    # But 'unaryOp' only looks up (Pred, None). 
    # If the data is (Napoleon, return, None), unaryOp works.
    # If data is (Napoleon, return, Paris), we must use binaryOp.
    
    # Let's inspect the dictionary directly for demonstration if needed, 
    # but let's try standard ops assuming some data is (S, P, None).
    # If not, we will likely get [0, 1] (False).
    
    # UPDATE: Based on exploration, 'return' often has None as object or specific location.
    # Let's check specific known facts from exploration: ('Napoleon', 'return', None) might exist?
    # Actually exploration showed: ('Napoleon', 'return', None) might not exist, but ('Napoleon', 'return', 'Paris') might.
    
    # To make this robust, let's just query what we found in the probabilities printout.
    # For the sake of the demo, I will use the specific objects if they appear in the printout.
    pass 

# Custom query function that adapts to what's in the data
def query_napoleon_logic(subj):
    # We will use specific objects if 'return' is binary.
    # From data exploration: ('Napoleon', 'return', None) was not explicitly shown, but let's try Unary first.
    # If 0 probability, we won't see much. 
    # However, for the demo, let's try to find the 'escape' and 'return' actions.
    
    # We can check the model's internal storage to see which 'return' fact was added.
    # But simpler: let's just try Unary. If 0, it means it was binary.
    
    p1 = "return"
    p2 = "escape"
    
    # Note: LogicModel unaryOp looks for (S, P). 
    val1 = model.unaryOp(p1, subj)
    val2 = model.unaryOp(p2, subj)
    
    print(f"\n1. {p1}({subj}): \n{val1.flatten()}")
    print(f"2. {p2}({subj}): \n{val2.flatten()}")
    print(f"3. {p1} AND {p2}: \n{model.andOp(val1, val2).flatten()}")

analyze_subject("Napoleon", ["return", "escape"], query_napoleon_logic)


--- Analysis: Napoleon ---
Max frequency count: 2


  Fact: return(Napoleon, None) | Count: 1 | Prob: 0.50
  Fact: escape(Napoleon, power) | Count: 1 | Prob: 0.50

1. return(Napoleon): 
[0.5 0.5]
2. escape(Napoleon): 
[0. 1.]
3. return AND escape: 
[0. 1.]


## Case Study 3: The Trio (Reagan, Chickasaw, Kaworu)

We analyze three diverse entities selected for this demo.

In [7]:
# Reagan
def query_reagan(subj):
    # signed (something), is_president (Unary)
    val1 = model.unaryOp("is_president", subj)
    # For 'signed', it's likely binary (signed a bill). Let's check Unary projection first.
    val2 = model.unaryOp("signed", subj)
    
    print(f"\n{subj} -> is_president: {val1.flatten()}")
    print(f"{subj} -> signed: {val2.flatten()}")
    print(f"Combined: {model.andOp(val1, val2).flatten()}")

analyze_subject("Reagan", ["is_president", "signed"], query_reagan)

# Chickasaw
def query_chickasaw(subj):
    val1 = model.unaryOp("is_people", subj)
    val2 = model.unaryOp("is_allies", subj)
    print(f"\n{subj} -> is_people: {val1.flatten()}")
    print(f"{subj} -> is_allies: {val2.flatten()}")
    print(f"Combined: {model.andOp(val1, val2).flatten()}")

analyze_subject("Chickasaw", ["is_people", "is_allies"], query_chickasaw)

# Kaworu
def query_kaworu(subj):
    val1 = model.unaryOp("plays", subj)
    val2 = model.unaryOp("is_apologizes", subj)
    print(f"\n{subj} -> plays: {val1.flatten()}")
    print(f"{subj} -> is_apologizes: {val2.flatten()}")
    print(f"Combined: {model.andOp(val1, val2).flatten()}")

analyze_subject("Kaworu", ["plays", "is_apologizes"], query_kaworu)


--- Analysis: Reagan ---
Max frequency count: 7
  Fact: signed(Reagan, Reform) | Count: 1 | Prob: 0.14


  Fact: is_president(Reagan, None) | Count: 1 | Prob: 0.14

Reagan -> is_president: [0.14285714 0.85714286]
Reagan -> signed: [0. 1.]
Combined: [0. 1.]

--- Analysis: Chickasaw ---
Max frequency count: 1
  Fact: is_people(Chickasaw, None) | Count: 1 | Prob: 1.00


  Fact: is_allies(Chickasaw, None) | Count: 1 | Prob: 1.00



Chickasaw -> is_people: [1. 0.]
Chickasaw -> is_allies: [1. 0.]
Combined: [1. 0.]

--- Analysis: Kaworu ---
Max frequency count: 2
  Fact: plays(Kaworu, role) | Count: 2 | Prob: 1.00


  Fact: is_apologizes(Kaworu, None) | Count: 1 | Prob: 0.50

Kaworu -> plays: [0. 1.]
Kaworu -> is_apologizes: [0.5 0.5]
Combined: [0. 1.]


## Case Study 4: Biology (Cells)
**Questions:**
*   Are cells blue?
*   Are cells coccoid (spherical)?
*   **Logic**: Blue AND Coccoid?

In [8]:
def query_cells(subj):
    val1 = model.unaryOp("is_blue", subj)
    val2 = model.unaryOp("is_coccoid", subj)
    
    print(f"\n1. is_blue: {val1.flatten()}")
    print(f"2. is_coccoid: {val2.flatten()}")
    print(f"3. Blue AND Coccoid: {model.andOp(val1, val2).flatten()}")

analyze_subject("cells", ["is_blue", "is_coccoid"], query_cells)


--- Analysis: cells ---
Max frequency count: 3
  Fact: is_coccoid(cells, None) | Count: 3 | Prob: 1.00
  Fact: is_blue(cells, None) | Count: 1 | Prob: 0.33

1. is_blue: [0.33333333 0.66666667]
2. is_coccoid: [1. 0.]
3. Blue AND Coccoid: [0.33333333 0.66666667]


## Case Study 5: Animals (Cats)
**Questions:**
*   Do cats jump?
*   Do cats land?
*   **Logic**: Jump AND Land?

In [9]:
def query_cats(subj):
    val1 = model.unaryOp("jump", subj)
    val2 = model.unaryOp("land", subj)
    
    print(f"\n1. jump: {val1.flatten()}")
    print(f"2. land: {val2.flatten()}")
    print(f"3. Jump AND Land: {model.andOp(val1, val2).flatten()}")

analyze_subject("cats", ["jump", "land"], query_cats)


--- Analysis: cats ---
Max frequency count: 2


  Fact: jump(cats, None) | Count: 1 | Prob: 0.50
  Fact: land(cats, None) | Count: 1 | Prob: 0.50



1. jump: [0.5 0.5]
2. land: [0.5 0.5]
3. Jump AND Land: [0.25 0.75]


## Case Study 6: War (Merged)
**Note:** We merged 'War' and 'war' during preprocessing.

**Questions:**
*   Did war begin?
*   Did war end?
*   **Logic**: Began AND Ended?

In [10]:
def query_war(subj):
    val1 = model.unaryOp("began", subj)
    val2 = model.unaryOp("ended", subj)
    
    print(f"\n1. began: {val1.flatten()}")
    print(f"2. ended: {val2.flatten()}")
    print(f"3. Began AND Ended: {model.andOp(val1, val2).flatten()}")

analyze_subject("war", ["began", "ended"], query_war)


--- Analysis: war ---
Max frequency count: 1
  Fact: began(war, cables) | Count: 1 | Prob: 1.00
  Fact: ended(war, None) | Count: 1 | Prob: 1.00


  Fact: ended(war, most) | Count: 1 | Prob: 1.00


  Fact: began(war, nutrition) | Count: 1 | Prob: 1.00
  Fact: ended(war, use) | Count: 1 | Prob: 1.00



1. began: [0. 1.]
2. ended: [1. 0.]
3. Began AND Ended: [0. 1.]
